# Evaluate & Finetune on Company Data
**Two independent models → weighted vote**

1. **wav2vec2** (ONNX) — audio-level read/spontaneous scoring
2. **Text-only XGBoost** — text + pause + prosodic features

This notebook:
- Auto-detects models and existing scores from common directories
- Evaluates pretrained models on audios2 and audios4 separately
- Shows weighted-vote combinations
- Finetunes XGBoost on company data (train/test split)
- Error analysis

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
import xgboost as xgb
import joblib

# ── Auto-detect model paths ──────────────────────────────────
def find_file(name, search_dirs):
    """Find a file in multiple candidate directories."""
    for d in search_dirs:
        p = Path(d) / name
        if p.exists():
            return str(p)
    return None

SEARCH_DIRS = [
    ".",                          # current dir
    "models",                     # models subfolder
    "checkpoints_finetuned",      # finetuned output
    "checkpoints_ensemble",       # ensemble checkpoints
    "checkpoints_ensemble_textonly",
    "checkpoints_combined",       # combined wav2vec2
    "checkpoints_biased",         # biased wav2vec2
    "biased", "ensemble", "ensemble_textonly", "combined",  # HF download subfolders
]

# wav2vec2 ONNX — prefer combined, fall back to biased
ONNX_MODEL = (
    find_file("wav2vec2_combined_quant.onnx", SEARCH_DIRS) or
    find_file("biased_wav2vec2_quant.onnx", SEARCH_DIRS)
)

# Text-only XGBoost (no wav2vec2 features) — prefer text-only, fall back to full ensemble
XGBOOST_MODEL = (
    find_file("xgboost_ensemble.json", [".", "models", "checkpoints_ensemble_textonly", "ensemble_textonly"]) or
    find_file("xgboost_ensemble.json", SEARCH_DIRS)
)
SCALER_PKL = (
    find_file("scaler.pkl", [".", "models", "checkpoints_ensemble_textonly", "ensemble_textonly"]) or
    find_file("scaler.pkl", SEARCH_DIRS)
)

# Features CSV — auto-detect
FEATURES_CSV = (
    find_file("features_company.csv", [".", "datasets"]) or
    find_file("features.csv", [".", "datasets"])
)

print("=== Auto-detected paths ===")
print(f"  ONNX model:    {ONNX_MODEL or 'NOT FOUND'}")
print(f"  XGBoost model: {XGBOOST_MODEL or 'NOT FOUND'}")
print(f"  Scaler:        {SCALER_PKL or 'NOT FOUND'}")
print(f"  Features CSV:  {FEATURES_CSV or 'NOT FOUND'}")

# Override any path here if auto-detect got it wrong:
# ONNX_MODEL = "path/to/model.onnx"
# XGBOOST_MODEL = "path/to/xgboost_ensemble.json"
# SCALER_PKL = "path/to/scaler.pkl"
# FEATURES_CSV = "path/to/features.csv"

## 1. Load Features & Labels

In [ ]:
# Feature group definitions
TEXT_FEATURES = [
    "filler_rate", "filler_count", "repetition_rate", "repair_rate",
    "ttr", "mattr", "complex_word_rate", "avg_word_length",
    "n_words", "n_unique_words",
    "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
    "self_ref_rate", "discourse_marker_rate", "hedge_rate",
    "noun_rate", "verb_rate", "adj_rate",
]
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
WAV2VEC2_FEATURES = [
    "wav2vec2_read_ratio", "wav2vec2_mean_p_read", "wav2vec2_max_p_read",
]
KNOWN_FEATURES = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES + WAV2VEC2_FEATURES
TEXT_ONLY_FEATURES = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES

# Metadata columns to exclude from auto-detection
META_COLS = {"filepath", "filename", "candidate", "audio_batch", "label", "label_raw",
             "label_int", "duration_sec", "n_transcript_words", "text", "split", "folder"}

def get_available_features(df, feature_list):
    """Return features that exist in df and have non-zero variance."""
    return [c for c in feature_list if c in df.columns and df[c].fillna(0).std() > 1e-8]

def detect_extra_features(df):
    """Auto-detect numeric columns not in any known feature list (e.g. mtld, etc.)."""
    extra = []
    for col in df.columns:
        if col in KNOWN_FEATURES or col in META_COLS:
            continue
        if df[col].dtype in ("float64", "float32", "int64", "int32"):
            if df[col].fillna(0).std() > 1e-8:
                extra.append(col)
    return extra

# Load features
df = pd.read_csv(FEATURES_CSV)
df = df[df["label_int"].isin([0, 1])].reset_index(drop=True)

print(f"Loaded {len(df)} labelled samples from {FEATURES_CSV}")
print(f"  Cheating (1):     {(df['label_int']==1).sum()}")
print(f"  Not cheating (0): {(df['label_int']==0).sum()}")

# Check for audio_batch column
if "audio_batch" in df.columns:
    print(f"\nBy batch:")
    for batch in sorted(df["audio_batch"].unique()):
        sub = df[df["audio_batch"] == batch]
        print(f"  {batch}: {len(sub)} ({(sub['label_int']==1).sum()} cheating, {(sub['label_int']==0).sum()} not)")

# Auto-detect extra features (e.g. mtld, mattr variants from senior)
EXTRA_FEATURES = detect_extra_features(df)
if EXTRA_FEATURES:
    print(f"\nAuto-detected {len(EXTRA_FEATURES)} EXTRA features (not in standard list):")
    for c in EXTRA_FEATURES:
        print(f"  [EXTRA   ] {c}  (mean={df[c].mean():.4f}, std={df[c].std():.4f})")
else:
    print(f"\nNo extra features detected beyond the standard {len(KNOWN_FEATURES)}.")

# Check for existing wav2vec2 scores
has_w2v_scores = "wav2vec2_mean_p_read" in df.columns and df["wav2vec2_mean_p_read"].sum() > 0
print(f"\nExisting wav2vec2 scores in CSV: {'YES' if has_w2v_scores else 'NO'}")

df.head()

## 2. wav2vec2 Scores (reuse existing or compute fresh)
If scores already exist in the features CSV, this cell skips re-computation. Set `FORCE_RESCORE = True` to override.

In [ ]:
FORCE_RESCORE = False

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def score_wav2vec2_onnx(df, onnx_path, sr=16000, window_sec=5.0, threshold=0.65):
    """Score all audio files with wav2vec2 ONNX. Returns (read_ratio, mean_p, max_p) arrays."""
    import onnxruntime as ort
    import librosa
    from tqdm import tqdm

    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    window = int(window_sec * sr)
    ratios, means, maxes = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring wav2vec2"):
        fp = row.get("filepath", "")
        if not fp or not os.path.exists(fp):
            ratios.append(0.0); means.append(0.0); maxes.append(0.0)
            continue
        try:
            audio, _ = librosa.load(fp, sr=sr, mono=True)
        except Exception:
            ratios.append(0.0); means.append(0.0); maxes.append(0.0)
            continue

        if len(audio) < window:
            ratios.append(0.0); means.append(0.0); maxes.append(0.0)
            continue

        p_reads = []
        for start in range(0, len(audio) - window + 1, window):
            chunk = audio[start:start+window].astype(np.float32).reshape(1, -1)
            logits = sess.run(None, {"input_values": chunk})[0]
            p_reads.append(sigmoid(logits.flatten()[0]))

        if p_reads:
            read_count = sum(1 for p in p_reads if p >= threshold)
            ratios.append(round(read_count / len(p_reads), 4))
            means.append(round(float(np.mean(p_reads)), 4))
            maxes.append(round(float(np.max(p_reads)), 4))
        else:
            ratios.append(0.0); means.append(0.0); maxes.append(0.0)

    return np.array(ratios), np.array(means), np.array(maxes)


if has_w2v_scores and not FORCE_RESCORE:
    print("Using existing wav2vec2 scores from features CSV (set FORCE_RESCORE=True to recompute)")
    w2v_proba = df["wav2vec2_mean_p_read"].values
else:
    if ONNX_MODEL:
        print(f"Computing wav2vec2 scores with: {ONNX_MODEL}")
        ratios, means, maxes = score_wav2vec2_onnx(df, ONNX_MODEL)
        df["wav2vec2_read_ratio"] = ratios
        df["wav2vec2_mean_p_read"] = means
        df["wav2vec2_max_p_read"] = maxes
        w2v_proba = means
        # Save updated CSV
        df.to_csv(FEATURES_CSV, index=False)
        print(f"  Saved updated scores to {FEATURES_CSV}")
    else:
        print("WARNING: No ONNX model found. wav2vec2 scores will be zero.")
        w2v_proba = np.zeros(len(df))

print(f"\nwav2vec2 score distribution:")
print(f"  mean={w2v_proba.mean():.4f}, std={w2v_proba.std():.4f}, min={w2v_proba.min():.4f}, max={w2v_proba.max():.4f}")

## 3. Text-only XGBoost Scores (pretrained)

In [ ]:
# Load pretrained text-only XGBoost
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(XGBOOST_MODEL)
scaler = joblib.load(SCALER_PKL)
print(f"Loaded XGBoost: {XGBOOST_MODEL}")
print(f"Loaded scaler:  {SCALER_PKL}")

# Get feature columns the model was trained on
text_feature_cols = get_available_features(df, TEXT_ONLY_FEATURES)
print(f"  Available text features: {len(text_feature_cols)}/{len(TEXT_ONLY_FEATURES)}")

# Fill missing features with 0
for c in TEXT_ONLY_FEATURES:
    if c not in df.columns:
        df[c] = 0

X_text = df[TEXT_ONLY_FEATURES].fillna(0).values
X_text_scaled = scaler.transform(X_text)
text_proba = xgb_model.predict_proba(X_text_scaled)[:, 1]
text_preds = (text_proba >= 0.5).astype(int)

print(f"\nText XGBoost score distribution:")
print(f"  mean={text_proba.mean():.4f}, std={text_proba.std():.4f}")

## 4. Evaluate Pretrained Models — per batch (audios2 vs audios4)

In [ ]:
y = df["label_int"].values
w2v_preds = (w2v_proba >= 0.5).astype(int)

def eval_print(y_true, y_pred, y_proba, name):
    if len(np.unique(y_true)) < 2:
        print(f"  [{name}] Only one class present, skipping")
        return {}
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_proba)
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  [{name}]")
    print(f"  Acc={acc:.4f}  F1={f1:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  AUC={auc:.4f}")
    if cm.shape == (2, 2):
        print(f"  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")
    return {"acc": acc, "f1": f1, "prec": prec, "rec": rec, "auc": auc}

# Evaluate per batch
batches = sorted(df["audio_batch"].unique()) if "audio_batch" in df.columns else ["all"]
results_per_batch = {}

for batch in batches:
    mask = (df["audio_batch"] == batch).values if "audio_batch" in df.columns else np.ones(len(df), dtype=bool)
    y_b = y[mask]
    if len(y_b) == 0:
        continue

    print(f"\n{'='*60}")
    print(f"{batch.upper()} ({len(y_b)} samples: {y_b.sum()} cheating, {(1-y_b).sum():.0f} not)")
    print(f"{'='*60}")

    r = {}
    r["wav2vec2"] = eval_print(y_b, w2v_preds[mask], w2v_proba[mask], "wav2vec2 (pretrained)")
    r["text_xgb"] = eval_print(y_b, text_preds[mask], text_proba[mask], "Text-only XGBoost (pretrained)")

    # Combined at different weights
    print(f"\n  [Combined weighted vote]")
    for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
        c_proba = w * w2v_proba[mask] + (1-w) * text_proba[mask]
        c_preds = (c_proba >= 0.5).astype(int)
        c_f1 = f1_score(y_b, c_preds, zero_division=0)
        c_acc = accuracy_score(y_b, c_preds)
        print(f"    w2v={w:.1f}: Acc={c_acc:.4f}, F1={c_f1:.4f}")

    results_per_batch[batch] = r

# Overall
print(f"\n{'='*60}")
print(f"OVERALL ({len(y)} samples)")
print(f"{'='*60}")
eval_print(y, w2v_preds, w2v_proba, "wav2vec2")
eval_print(y, text_preds, text_proba, "Text-only XGBoost")

print(f"\n  [Combined — weight sensitivity]")
best_w, best_f1 = 0.5, 0
for w in np.arange(0.1, 0.91, 0.1):
    c_proba = w * w2v_proba + (1-w) * text_proba
    c_preds = (c_proba >= 0.5).astype(int)
    c_f1 = f1_score(y, c_preds, zero_division=0)
    c_acc = accuracy_score(y, c_preds)
    marker = ""
    if c_f1 > best_f1:
        best_f1 = c_f1
        best_w = w
        marker = " <-- best"
    print(f"    w2v={w:.1f}: Acc={c_acc:.4f}, F1={c_f1:.4f}{marker}")

print(f"\n  Best weight: w2v={best_w:.1f}, F1={best_f1:.4f}")

## 5. Finetune XGBoost on Company Data (text-only — NO wav2vec2 features)
Trains on audios2 + audios4 combined with 80/20 stratified split. Text + pause + prosodic only. wav2vec2 stays separate for weighted vote.

In [ ]:
# Text-only features — NO wav2vec2 (it's a separate voter)
# Includes any extra features auto-detected from the CSV (e.g. mtld)
ALL_FEATURES = get_available_features(df, TEXT_ONLY_FEATURES) + EXTRA_FEATURES
print(f"Finetuning with {len(ALL_FEATURES)} features (wav2vec2 excluded):")
for c in ALL_FEATURES:
    group = ("TEXT" if c in TEXT_FEATURES else "PAUSE" if c in PAUSE_FEATURES
             else "PROSODIC" if c in PROSODIC_FEATURES else "EXTRA")
    print(f"  [{group:8s}] {c}")

X = df[ALL_FEATURES].fillna(0).values

# Train/test split — stratified
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(len(y)), test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {len(y_train)} ({y_train.sum()} cheating, {(1-y_train).sum():.0f} not)")
print(f"Test:  {len(y_test)} ({y_test.sum()} cheating, {(1-y_test).sum():.0f} not)")

# Show batch distribution
if "audio_batch" in df.columns:
    for name, idxs in [("Train", idx_train), ("Test", idx_test)]:
        counts = df.iloc[idxs]["audio_batch"].value_counts().to_dict()
        print(f"  {name} batches: {counts}")

# Scale
ft_scaler = StandardScaler()
X_train_s = ft_scaler.fit_transform(X_train)
X_test_s = ft_scaler.transform(X_test)

# Train
ft_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=42,
)

ft_model.fit(X_train_s, y_train,
             eval_set=[(X_train_s, y_train), (X_test_s, y_test)],
             verbose=20)

# CV on train
cv_scores = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric="logloss"),
    X_train_s, y_train, cv=5, scoring="f1"
)
print(f"\nCV F1 (train): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

## 6. Finetuned Results + Feature Importance (text-only XGBoost)

In [ ]:
# Test set results
ft_proba = ft_model.predict_proba(X_test_s)[:, 1]
ft_preds = (ft_proba >= 0.5).astype(int)

print("="*60)
print("FINETUNED XGBoost — Test Set")
print("="*60)
eval_print(y_test, ft_preds, ft_proba, "Finetuned XGBoost (all features)")
print(f"\n{classification_report(y_test, ft_preds, target_names=['not cheating', 'cheating'])}")

# Feature importance
importances = ft_model.feature_importances_
feat_imp = sorted(zip(ALL_FEATURES, importances), key=lambda x: -x[1])

print("Feature Importance (top 20):")
for name, imp in feat_imp[:20]:
    bar = "#" * int(imp * 100)
    print(f"  {name:<35s} {imp:.4f} {bar}")

# Threshold analysis
print(f"\nThreshold Sensitivity:")
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    t_pred = (ft_proba >= thresh).astype(int)
    t_prec = precision_score(y_test, t_pred, zero_division=0)
    t_rec = recall_score(y_test, t_pred, zero_division=0)
    t_f1 = f1_score(y_test, t_pred, zero_division=0)
    marker = " <-- default" if thresh == 0.5 else ""
    print(f"  threshold={thresh:.1f}: prec={t_prec:.4f} rec={t_rec:.4f} f1={t_f1:.4f}{marker}")

## 7. Combined: Finetuned XGBoost + Pretrained wav2vec2 (weighted vote)
Compare the finetuned XGBoost alone vs combined with wav2vec2 at various weights.

In [ ]:
# Get finetuned XGBoost probabilities on test set
# Also get wav2vec2 probabilities for the same test indices
test_w2v = w2v_proba[idx_test]

print("="*60)
print("COMBINED VOTE: Finetuned XGBoost + wav2vec2 (test set)")
print("="*60)

print(f"\n  Finetuned XGBoost alone:  F1={f1_score(y_test, ft_preds, zero_division=0):.4f}")
print(f"  wav2vec2 alone:           F1={f1_score(y_test, (test_w2v >= 0.5).astype(int), zero_division=0):.4f}")

print(f"\n  Weighted combinations:")
best_combo_w, best_combo_f1 = 0, 0
for w in np.arange(0.0, 1.01, 0.1):
    c_proba = w * test_w2v + (1 - w) * ft_proba
    c_preds = (c_proba >= 0.5).astype(int)
    c_f1 = f1_score(y_test, c_preds, zero_division=0)
    c_acc = accuracy_score(y_test, c_preds)
    marker = ""
    if c_f1 > best_combo_f1:
        best_combo_f1 = c_f1
        best_combo_w = w
        marker = " <-- best"
    label = "xgb only" if w == 0 else "w2v only" if w == 1.0 else f"w2v={w:.1f}"
    print(f"    {label:>12s}: Acc={c_acc:.4f}, F1={c_f1:.4f}{marker}")

print(f"\n  Best: w2v={best_combo_w:.1f} + xgb={1-best_combo_w:.1f}, F1={best_combo_f1:.4f}")

# Show classification report for best combo
best_c_proba = best_combo_w * test_w2v + (1 - best_combo_w) * ft_proba
best_c_preds = (best_c_proba >= 0.5).astype(int)
print(f"\n{classification_report(y_test, best_c_preds, target_names=['not cheating', 'cheating'])}")

## 7b. Threshold Selection + 5-Fold CV
Fine-grained threshold sweep for the best combined model, plus cross-validated stability estimate.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# ── Threshold sweep on combined scores (test set) ─────────────
test_w2v = w2v_proba[idx_test]
best_combined = best_combo_w * test_w2v + (1 - best_combo_w) * ft_proba

thresholds = np.arange(0.10, 0.91, 0.05)
rows_t = []
for t in thresholds:
    preds = (best_combined >= t).astype(int)
    rows_t.append({
        "threshold": round(t, 2),
        "precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "f1": round(f1_score(y_test, preds, zero_division=0), 4),
        "flagged": int(preds.sum()),
        "missed": int(((y_test == 1) & (preds == 0)).sum()),
        "false_alarms": int(((y_test == 0) & (preds == 1)).sum()),
    })

thresh_df = pd.DataFrame(rows_t)
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]

print(f"Combined model threshold sweep (w2v={best_combo_w:.1f}, xgb={1-best_combo_w:.1f})")
print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'missed':>7s} {'false_alarm':>11s}")
print("-" * 62)
for _, r in thresh_df.iterrows():
    marker = " <-- best F1" if r["threshold"] == best_row["threshold"] else ""
    print(f"  {r['threshold']:.2f}   {r['precision']:.4f}  {r['recall']:.4f}  {r['f1']:.4f}  {r['flagged']:>6d}  {r['missed']:>6d}  {r['false_alarms']:>6d}{marker}")

print(f"\nBest F1 at threshold={best_row['threshold']:.2f}: F1={best_row['f1']:.4f}, Prec={best_row['precision']:.4f}, Rec={best_row['recall']:.4f}")

# ── 5-Fold CV for stability estimate ──────────────────────────
print(f"\n{'='*60}")
print("5-FOLD CROSS-VALIDATION (text-only XGBoost)")
print(f"{'='*60}")

X_all = df[ALL_FEATURES].fillna(0).values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
for fold, (tr_idx, te_idx) in enumerate(skf.split(X_all, y), 1):
    sc = StandardScaler()
    X_tr = sc.fit_transform(X_all[tr_idx])
    X_te = sc.transform(X_all[te_idx])

    m = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y[tr_idx] == 0).sum() / max((y[tr_idx] == 1).sum(), 1),
        eval_metric="logloss", random_state=42,
    )
    m.fit(X_tr, y[tr_idx], verbose=False)

    xgb_prob = m.predict_proba(X_te)[:, 1]
    w2v_fold = w2v_proba[te_idx]
    combined_fold = best_combo_w * w2v_fold + (1 - best_combo_w) * xgb_prob

    # Evaluate at best threshold
    t = best_row["threshold"]
    preds_fold = (combined_fold >= t).astype(int)
    f = f1_score(y[te_idx], preds_fold, zero_division=0)
    p = precision_score(y[te_idx], preds_fold, zero_division=0)
    r = recall_score(y[te_idx], preds_fold, zero_division=0)
    fold_results.append({"fold": fold, "f1": f, "precision": p, "recall": r})
    print(f"  Fold {fold}: F1={f:.4f}  Prec={p:.4f}  Rec={r:.4f}")

fold_df = pd.DataFrame(fold_results)
print(f"\n  Mean:  F1={fold_df['f1'].mean():.4f} +/- {fold_df['f1'].std():.4f}")
print(f"         Prec={fold_df['precision'].mean():.4f} +/- {fold_df['precision'].std():.4f}")
print(f"         Rec={fold_df['recall'].mean():.4f} +/- {fold_df['recall'].std():.4f}")

# ── Set your chosen threshold here ────────────────────────────
CHOSEN_THRESHOLD = best_row["threshold"]
print(f"\n>> Using threshold={CHOSEN_THRESHOLD:.2f} for predictions. Edit CHOSEN_THRESHOLD to change.")

## 8. Save Finetuned Model

In [ ]:
os.makedirs("checkpoints_finetuned", exist_ok=True)

ft_model.save_model("checkpoints_finetuned/xgboost_finetuned.json")
joblib.dump(ft_scaler, "checkpoints_finetuned/scaler_finetuned.pkl")

config = {
    "feature_columns": ALL_FEATURES,
    "best_wav2vec2_weight": round(float(best_combo_w), 2),
    "finetuned_f1": round(f1_score(y_test, ft_preds, zero_division=0), 4),
    "combined_f1": round(float(best_combo_f1), 4),
    "cv_f1_mean": round(cv_scores.mean(), 4),
    "n_train": len(y_train),
    "n_test": len(y_test),
    "onnx_model_used": ONNX_MODEL or "none",
    "feature_importances": {name: round(float(imp), 6) for name, imp in feat_imp},
}
with open("checkpoints_finetuned/results_finetuned.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved to checkpoints_finetuned/:")
print("  xgboost_finetuned.json   - finetuned XGBoost model")
print("  scaler_finetuned.pkl     - feature scaler")
print("  results_finetuned.json   - results + config")

## 9. Error Analysis

In [ ]:
# Error analysis on test set using best combined model
test_df = df.iloc[idx_test].copy()
test_df["pred_prob"] = best_c_proba
test_df["pred_label"] = best_c_preds
test_df["correct"] = test_df["label_int"].values == best_c_preds
test_df["w2v_score"] = test_w2v
test_df["xgb_score"] = ft_proba

errors = test_df[~test_df["correct"]]
print(f"Errors: {len(errors)} / {len(test_df)} ({100*len(errors)/len(test_df):.1f}%)")

if len(errors) > 0:
    fp = errors[errors["pred_label"] == 1]  # predicted cheating, actually not
    fn = errors[errors["pred_label"] == 0]  # predicted not cheating, actually cheating

    print(f"\n  False Positives (flagged innocent): {len(fp)}")
    print(f"  False Negatives (missed cheater):   {len(fn)}")

    for label, subset in [("FP (innocent flagged)", fp), ("FN (cheater missed)", fn)]:
        if len(subset) == 0:
            continue
        print(f"\n  --- {label} ---")
        print(f"  wav2vec2 mean score: {subset['w2v_score'].mean():.4f}")
        print(f"  XGBoost mean score:  {subset['xgb_score'].mean():.4f}")
        print(f"  Models agree:        {(subset['w2v_score'] >= 0.5).astype(int).eq((subset['xgb_score'] >= 0.5).astype(int)).mean():.1%}")

        for col in ["filler_rate", "ttr", "hedge_rate", "pause_ratio", "wav2vec2_read_ratio"]:
            if col in subset.columns:
                print(f"    {col}: {subset[col].mean():.4f}")

    # Show individual errors
    show_cols = ["filename", "label_int", "pred_label", "w2v_score", "xgb_score", "pred_prob"]
    if "audio_batch" in errors.columns:
        show_cols.insert(1, "audio_batch")
    print(f"\nAll errors:")
    display(errors[show_cols].sort_values("pred_prob", ascending=False)) if len(errors) <= 50 else print(errors[show_cols].head(30).to_string())
else:
    print("No errors! Perfect classification.")

## 10. Predict on All Data (using best combined approach)

In [ ]:
# Predict on ALL samples using finetuned XGBoost + wav2vec2 weighted vote
W2V_WEIGHT = best_combo_w  # from cell 7
THRESHOLD = CHOSEN_THRESHOLD  # from cell 7b

X_all_scaled = ft_scaler.transform(df[ALL_FEATURES].fillna(0).values)
xgb_all_proba = ft_model.predict_proba(X_all_scaled)[:, 1]
combined_all_proba = W2V_WEIGHT * w2v_proba + (1 - W2V_WEIGHT) * xgb_all_proba

df["xgb_score"] = xgb_all_proba
df["w2v_score"] = w2v_proba
df["combined_score"] = combined_all_proba
df["pred_label"] = (combined_all_proba >= THRESHOLD).astype(int)
df["pred_label_str"] = df["pred_label"].map({1: "cheating", 0: "not cheating"})

# Short audio fallback: < 20 words → wav2vec2 only
if "n_words" in df.columns:
    short = df["n_words"] < 20
    if short.sum() > 0:
        print(f"Short audio fallback ({short.sum()} files < 20 words): using wav2vec2 only")
        df.loc[short, "combined_score"] = df.loc[short, "w2v_score"]
        df.loc[short, "pred_label"] = (df.loc[short, "w2v_score"] >= THRESHOLD).astype(int)
        df.loc[short, "pred_label_str"] = df.loc[short, "pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"\nPredictions ({len(df)} files):")
print(f"  Cheating:     {(df['pred_label']==1).sum()}")
print(f"  Not cheating: {(df['pred_label']==0).sum()}")
print(f"  Weights: w2v={W2V_WEIGHT:.1f}, xgb={1-W2V_WEIGHT:.1f}")
print(f"  Threshold: {THRESHOLD:.2f}")

# Save
output_cols = ["filename", "filepath", "label_int", "pred_label_str", "combined_score",
               "w2v_score", "xgb_score", "wav2vec2_read_ratio",
               "filler_rate", "hedge_rate", "pause_ratio", "n_words"]
if "audio_batch" in df.columns:
    output_cols.insert(2, "audio_batch")
output_cols = [c for c in output_cols if c in df.columns]

df[output_cols].to_csv("predictions.csv", index=False)
print(f"\nSaved: predictions.csv")
df[output_cols].head(10)